# 04.06 · Decoding: temperature, top-k, top-p, logprobs

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] Implement greedy, temperature, top-k, top-p (nucleus) and min-p sampling on real logits
- [ ] Know what the API parameters `temperature`, `top_p`, `max_tokens`, `stop`, `seed` actually do
- [ ] Use token log-probabilities for confidence scores and zero-shot classification

## Resources
- [HF – How to generate text](https://huggingface.co/blog/how-to-generate) — free
- [The Curious Case of Neural Text Degeneration (top-p paper)](https://arxiv.org/abs/1904.09751) — free
- [OpenAI cookbook – Using logprobs](https://cookbook.openai.com/examples/using_logprobs) — free

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()


def prompt_ids(user_text):
    messages = [{"role": "user", "content": user_text}]
    return tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(device)


@torch.no_grad()
def next_token_logits(ids):
    return model(ids).logits[0, -1].float()  # logits for the next position, shape (vocab,)

## 1. Look at the distribution

In [ ]:
ids = prompt_ids("Name a fruit.")
logits = next_token_logits(ids)
probs = logits.softmax(-1)
top = probs.topk(10)
for p, i in zip(top.values, top.indices):
    print(f"{p:.3f} {tok.decode(i)!r}")

## 2. Sampling strategies, implemented
All of these just reshape the distribution before `torch.multinomial`:
- **greedy**: argmax. Deterministic, often repetitive.
- **temperature**: divide logits by T (01.04).
- **top-k**: keep the k most likely tokens.
- **top-p (nucleus)**: keep the smallest set whose cumulative probability ≥ p. Adapts to how confident the model is.
- **min-p**: keep tokens with probability ≥ `min_p × max_prob`. Popular in local inference.

In [ ]:
def filter_logits(logits, temperature=1.0, top_k=None, top_p=None, min_p=None):
    logits = logits / max(temperature, 1e-6)
    if top_k is not None:
        kth = logits.topk(top_k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:
        sorted_logits, sorted_idx = logits.sort(descending=True)
        cum = sorted_logits.softmax(-1).cumsum(-1)
        remove = cum - sorted_logits.softmax(-1) > top_p  # drop tokens once the mass *before* them exceeds p
        logits = logits.masked_fill(remove.scatter(0, sorted_idx, remove), float("-inf"))
    if min_p is not None:
        p = logits.softmax(-1)
        logits = logits.masked_fill(p < min_p * p.max(), float("-inf"))
    return logits


for name, kw in {"T=1": {}, "T=0.5": {"temperature": 0.5}, "top_k=5": {"top_k": 5},
                 "top_p=0.9": {"top_p": 0.9}, "min_p=0.1": {"min_p": 0.1}}.items():
    kept = torch.isfinite(filter_logits(logits, **kw)).sum().item()
    print(f"{name:10} candidate tokens: {kept}")

In [ ]:
@torch.no_grad()
def generate(user_text, max_new_tokens=60, greedy=False, seed=0, **filters):
    torch.manual_seed(seed)
    ids = prompt_ids(user_text)
    for _ in range(max_new_tokens):
        logits = next_token_logits(ids)
        nxt = logits.argmax() if greedy else torch.multinomial(filter_logits(logits, **filters).softmax(-1), 1)[0]
        if nxt.item() == tok.eos_token_id:
            break
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return tok.decode(ids[0, prompt_ids(user_text).shape[1]:])


q = "Write one sentence about the ocean."
print("greedy     :", generate(q, greedy=True))
for kw in [{"temperature": 0.7, "top_p": 0.9}, {"temperature": 1.5}, {"temperature": 1.5, "min_p": 0.1}]:
    print(f"{str(kw):33}:", generate(q, **kw))

High temperature alone produces garbage; min-p/top-p cut the long tail of junk tokens, so you can use higher temperatures
for creativity while staying coherent.

**Which API params map to this?** `temperature`, `top_p`, sometimes `top_k`/`min_p` (local engines), `max_tokens` (loop length),
`stop` (end early on strings), `seed` (reproducibility, best-effort on APIs). Reasoning models often fix or ignore temperature.

## 3. Beam search
Keep the `n` best partial sequences by total log-prob. Good for translation/short exact outputs; bland for open-ended chat.

In [ ]:
ids = prompt_ids(q)
out = model.generate(ids, max_new_tokens=40, num_beams=4, do_sample=False, pad_token_id=tok.eos_token_id)
print("beam=4:", tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))

## 4. Logprobs: confidence and classification
The log-probability of a token tells you how confident the model was. Two powerful uses:
1. **Zero-shot classification**: compare the probability of each label token. No parsing, gives a real score.
2. **Confidence**: flag low-probability answers for review. Many APIs return `logprobs` (`logprobs=True, top_logprobs=5` in OpenAI-style APIs).

In [ ]:
def classify(text, labels=("positive", "negative")):
    ids = prompt_ids(f"Is the sentiment of this review positive or negative? Answer with one word.\n\nReview: {text}")
    logp = next_token_logits(ids).log_softmax(-1)
    # first token of each label (with and without capital) — check that labels start with distinct tokens!
    scores = {lab: torch.logsumexp(torch.stack([logp[tok.encode(v)[0]] for v in (lab, lab.capitalize())]), 0).item() for lab in labels}
    total = torch.logsumexp(torch.tensor(list(scores.values())), 0).item()
    return {lab: round(torch.exp(torch.tensor(s - total)).item(), 3) for lab, s in scores.items()}


for review in ["Absolutely loved it, would buy again.", "Broke after two days. Waste of money.", "It's okay I guess."]:
    print(f"{review:40} {classify(review)}")

A 135M model is a weak classifier — "loved it" may come out near 50/50. That's the point of measuring: logprobs tell you *how*
unsure the model is, so you can route low-confidence cases to a bigger model or a human. Try `Qwen/Qwen3-0.6B` and compare.

## Exercises
1. Load your tiny GPT checkpoint (`data/tinygpt.pt` from 04.03) and add top-p and min-p to its `generate`.
2. Measure **repetition**: generate 10 samples of 100 tokens with greedy, T=0.7+top_p=0.9 and T=1.5; count repeated 4-grams in each.
3. Turn `classify` into a 3-class classifier (positive / negative / neutral) and evaluate it on 15 reviews you write. Then compare with
   asking the model to answer in text and parsing it.

In [ ]:
# your code here